In [ ]:
pip install pandas requests sqlalchemy pyodbc pyarrow fastparquet

In [ ]:
pip install openlocationcode

In [5]:
pip install groq

In [30]:
import os

import pandas as pd
from sqlalchemy import create_engine
from openlocationcode import openlocationcode as olc
import numpy as np
import re
from sqlalchemy.types import NVARCHAR, Float
from google import genai
import json
from math import ceil
from groq import Groq
import requests
import time


category = ["clubs", "gas_stations", "hospitals", "schools", "universities"]

for c in category:
    server = "localhost"
    database = "staging_c_db"

    connection_string = (
        f"mssql+pyodbc://@{server}/{database}"
        "?driver=ODBC+Driver+18+for+SQL+Server"
        "&TrustServerCertificate=yes"
    )

    staging_engine = create_engine(connection_string)

    server = "localhost"
    database = "cleaning_c_db"

    connection_string = (
        f"mssql+pyodbc://@{server}/{database}"
        "?driver=ODBC+Driver+18+for+SQL+Server"
        "&TrustServerCertificate=yes"
    )

    cleaning_engine = create_engine(connection_string)

    df = pd.read_sql(f"SELECT * FROM stg_{c}", staging_engine)


    arabic_to_english = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

    df["Reviews"] = (
        df["Reviews"]
        .str.replace("عدد التعليقات:", "")
        .str.replace("٬", "")
        .str.strip()
        .str.translate(arabic_to_english)
    )

    # Clean the "Reviews" and "Rating" columns
    df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce").astype("Int64")
    df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce").astype("float") 

    # Clean the "Phone" 
    df["Phone"] = df["Phone"].str.replace("?\n", "", regex=False)
    df["Phone"] = df["Phone"].str.replace(" ", "", regex=False)

    # Clean the "Location"
    df["Location"] = df["Location"].str.extract(r'([A-Z0-9]{4}\+[A-Z0-9]{2})')
    df["Location"] = df["Location"].fillna(
        df["Address"].str.extract(r'([A-Z0-9]{4}\+[A-Z0-9]{2})')[0]
    )

    def decode_pluscode_local(code):

        if pd.notna(code):
                ref_lat, ref_lng = 30.0444, 31.2357
                full_code = olc.recoverNearest(code, ref_lat, ref_lng)
                decoded = olc.decode(full_code)

                return pd.Series([decoded.latitudeCenter, decoded.longitudeCenter])

        else:
            return pd.Series([30.0444, 31.2357]) # Default to Cairo coordinates if code is missing


    df[["Latitude", "Longitude"]] = df["Location"].apply(decode_pluscode_local)

    #----------------------------------------------------------------------------------------API----------------------------------------------------------------


    def reverse_geocode(lat, lon):
        try:
            url = f"https://nominatim.openstreetmap.org/reverse"
            params = {
                "lat": lat,
                "lon": lon,
                "format": "json",
                "addressdetails": 1
            }
            headers = {"User-Agent": "MyApp/1.0"}
            
            response = requests.get(url, params=params, headers=headers, timeout=10)
            data = response.json()
            
            addr = data.get("address", {})
            
            # ✅ اطبع كل الـ fields اللي بترجع عشان نشوف الأسماء الصح
            print(addr)
            
            return {
                        "street":      addr.get("road") or addr.get("pedestrian") or addr.get("path") or addr.get("neighbourhood") ,
                        
                        "district":    addr.get("suburb") or addr.get("neighbourhood") or 
                                    addr.get("quarter") or addr.get("residential") or addr.get("city"),
                        
                        "region":      addr.get("city") or addr.get("city_district") or addr.get("borough") or 
                                    addr.get("municipality") or addr.get("town") or 
                                    addr.get("village") or addr.get("county")or addr.get("suburb"),
                        
                        "governorate": addr.get("state") or addr.get("province"),
                        "postcode": addr.get("postcode")
                    }
        except Exception as e:
            print(f"❌ Error for ({lat}, {lon}): {e}")
            return {"street": None, "district": None, "region": None, "governorate": None}


    # ── تطبيق على الـ DataFrame ──────────────────
    results = []

    for i, row in df.iterrows():
        result = reverse_geocode(row['Latitude'], row['Longitude'])
        results.append(result)
        time.sleep(1)  # ✅ مطلوب - Nominatim بيحظر لو بعت requests كتير بسرعة
        
        if i % 50 == 0:
            print(f"✅ {i}/{len(df)}")

    df['street']      = [r.get('street')      for r in results]
    df['district']    = [r.get('district')    for r in results]
    df['region']      = [r.get('region')      for r in results]
    df['governorate'] = [r.get('governorate') for r in results]
    df['postcode']    = [r.get('postcode')    for r in results]
 

    df.to_sql(
                f"clean_{c}",
                cleaning_engine,
                if_exists="replace",
                index=False,
                dtype={
                    "City": NVARCHAR(100),
                    "Category": NVARCHAR(100),
                    "Sub Category": NVARCHAR(200),
                    "English Name": NVARCHAR(200),
                    "Arabic Name": NVARCHAR(200),
                    "Reviews": NVARCHAR(200),
                    "Address": NVARCHAR(500),
                    "Rating": Float,
                    "street": NVARCHAR(None),
                    "district": NVARCHAR(None),
                    "region": NVARCHAR(None),  
                    "governorate": NVARCHAR(None)
                }
            )
print("Data has been cleaned and loaded into the cleaning database successfully!")


c:\Users\maged.magdy\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


{'leisure': 'نادي القاهرة الرياضي', 'road': 'شارع التحرير', 'suburb': 'باب اللوق', 'state': 'القاهرة', 'ISO3166-2-lvl4': 'EG-C', 'postcode': '11111', 'country': 'مصر', 'country_code': 'eg'}
✅ 0/120
{'leisure': 'نادي الجزيرة الرياضي', 'road': 'شارع الجزيره', 'suburb': 'بولاق', 'state': 'القاهرة', 'ISO3166-2-lvl4': 'EG-C', 'postcode': '11111', 'country': 'مصر', 'country_code': 'eg'}
{'leisure': 'نادي مدينة نصر الرياضي', 'road': 'شارع المخيم الدائم', 'city': 'ثان مدينة نصر', 'state': 'القاهرة', 'ISO3166-2-lvl4': 'EG-C', 'postcode': '11759', 'country': 'مصر', 'country_code': 'eg'}
{'club': 'نادي هليوبوليس الرياضى', 'house_number': '17', 'road': 'شارع السيد الميرغنى', 'quarter': 'قسم مصر الجديدة', 'suburb': 'مصر الجديدة', 'city': 'القاهرة', 'state': 'القاهرة', 'ISO3166-2-lvl4': 'EG-C', 'postcode': '11774', 'country': 'مصر', 'country_code': 'eg'}
{'leisure': 'نادي الزهور', 'road': 'شارع يوسف عباس', 'neighbourhood': 'رابعة العدوية', 'city': 'مدينة نصر', 'state': 'القاهرة', 'ISO3166-2-lvl4': '

c:\Users\maged.magdy\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())
c:\Users\maged.magdy\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


{'amenity': 'طاقا', 'road': 'شارع مجمع الفردوس', 'neighbourhood': 'مساكن مجمع الفردوس', 'city': 'ثان مدينة نصر', 'state': 'القاهرة', 'ISO3166-2-lvl4': 'EG-C', 'postcode': '11684', 'country': 'مصر', 'country_code': 'eg'}
✅ 0/120
{'road': 'شارع الخليفه المطيع', 'neighbourhood': 'التجمع الخامس', 'village': 'أحياء التجمع الخامس', 'city': 'القاهرة', 'state': 'القاهرة', 'ISO3166-2-lvl4': 'EG-C', 'postcode': '11835', 'country': 'مصر', 'country_code': 'eg'}
{'house_number': '1', 'road': 'شارع سعد بن ابي وقاص', 'neighbourhood': 'فم\xa0الخليج\xa0ودير\xa0النحاس', 'suburb': 'مصر القديمة', 'city': 'القاهرة', 'state': 'القاهرة', 'ISO3166-2-lvl4': 'EG-C', 'postcode': '11632', 'country': 'مصر', 'country_code': 'eg'}
{'amenity': 'Mobil', 'road': 'شارع صلاح سالم', 'suburb': 'عين الصيرة', 'city': 'القاهرة', 'state': 'القاهرة', 'ISO3166-2-lvl4': 'EG-C', 'postcode': '11654', 'country': 'مصر', 'country_code': 'eg'}
{'amenity': 'محطة بنزين توتال', 'road': 'طريق النصر', 'city': 'القاهرة', 'state': 'القاهرة', 

c:\Users\maged.magdy\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())
c:\Users\maged.magdy\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


...

{'house_number': '12B', 'road': 'شارع خالد بن الوليد', 'neighbourhood': 'محلية 12', 'city': 'العبور', 'state': 'القليوبية', 'ISO3166-2-lvl4': 'EG-KB', 'postcode': '11828', 'country': 'مصر', 'country_code': 'eg'}
✅ 0/120
{'road': 'ميدان التحرير', 'neighbourhood': 'قصر\xa0الدوباره', 'suburb': 'باب اللوق', 'city': 'القاهرة', 'state': 'القاهرة', 'ISO3166-2-lvl4': 'EG-C', 'postcode': '11519', 'country': 'مصر', 'country_code': 'eg'}
{'road': 'شارع ابراهيم عرفه', 'neighbourhood': 'الحى 7', 'city': 'العبور', 'state': 'القليوبية', 'ISO3166-2-lvl4': 'EG-KB', 'postcode': '02000', 'country': 'مصر', 'country_code': 'eg'}
{'road': 'شارع سيف الدين مهراني', 'neighbourhood': 'الفجاله', 'suburb': 'باب الشعرية', 'city': 'القاهرة', 'state': 'القاهرة', 'ISO3166-2-lvl4': 'EG-C', 'postcode': '11523', 'country': 'مصر', 'country_code': 'eg'}
{'amenity': 'المدرسة البريطانية', 'road': 'شارع دكتور احمد مصطفى احمد', 'neighbourhood': 'التجمع الخامس', 'village': 'أحياء التجمع الخامس', 'city': 'القاهرة', 'state': 'ال

c:\Users\maged.magdy\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())
c:\Users\maged.magdy\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


{'amenity': 'Misr International University', 'road': 'طريق القاهرة, الاسماعيليه الصحراوي', 'city': 'مدينتي', 'state': 'القليوبية', 'ISO3166-2-lvl4': 'EG-KB', 'postcode': '11785', 'country': 'مصر', 'country_code': 'eg'}
✅ 0/84
{'house_number': '1', 'neighbourhood': 'القطاع الرابع', 'village': 'أحياء اللوتس', 'city': 'القاهرة', 'state': 'القاهرة', 'ISO3166-2-lvl4': 'EG-C', 'postcode': '11865', 'country': 'مصر', 'country_code': 'eg'}
{'amenity': 'MUST Opera House', 'road': 'محور 26 يوليو الجانبى', 'neighbourhood': 'بلوك 3', 'suburb': 'الحي السابع', 'city': 'السادس من أكتوبر', 'state': 'الجيزة', 'ISO3166-2-lvl4': 'EG-GZ', 'postcode': '12568', 'country': 'مصر', 'country_code': 'eg'}
{'amenity': 'الجامعة البريطانية فى مصر', 'road': 'طريق الحريه', 'residential': 'مدينة الشروق', 'village': 'ماي فير', 'city': 'القاهرة', 'state': 'القاهرة', 'ISO3166-2-lvl4': 'EG-C', 'postcode': '19511', 'country': 'مصر', 'country_code': 'eg'}
{'road': 'كوبرى 6 أكتوبر', 'suburb': 'الدمرداش', 'village': 'الدمرداش'

c:\Users\maged.magdy\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\io\sql.py:1649: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Data has been cleaned and loaded into the cleaning database successfully!
